# Course: Introduction to GPU Programming in Python

**Duration:** 3 Hours  
**Target Audience:** Undergraduate Data Science Students  
**Prerequisites:** Basic Python, NumPy, Data Science concepts

## 📌 Session Overview

This hands-on session introduces the fundamentals of GPU acceleration for data science using Python. We focus on **high-level tools** (CuPy and Numba) that allow you to accelerate existing code without learning low-level C++/CUDA.

### 🕒 Time Allocation Strategy

| Block | Duration | Topic | Activity Type |
|---|---|---|---|
| **1** | 0:00 - 0:20 | **Concepts & Setup:** Why GPU? Latency vs. Throughput | Lecture & Hardware Check |
| **2** | 0:20 - 1:00 | **CuPy:** NumPy on Steroids | Live Demo & Benchmarking |
| **3** | 1:00 - 1:15 | **The Bottleneck:** Data Transfer Overhead | Failure Analysis Demo |
| **4** | 1:15 - 1:30 | **Break** | - |
| **5** | 1:30 - 2:15 | **Numba:** JIT Compilation for GPUs | Guided Coding |
| **6** | 2:15 - 2:45 | **Capstone:** Monte Carlo Simulation | Hands-on Exercise |
| **7** | 2:45 - 3:00 | **Wrap-up:** Best Practices & Q&A | Discussion |

---

## 🛠️ Environment Setup
First, we ensure our Google Colab environment has a GPU allocated.

In [ ]:
# Check NVIDIA System Management Interface for GPU details
!nvidia-smi

# Block 1: Concepts & Intuition (20 Mins)

## 🧠 Theory: The CPU vs. GPU Analogy

### The CPU (Central Processing Unit)
* **Analogy:** A Ferrari.
* **Strength:** Extremely fast at doing *one thing* at a time (Low Latency).
* **Best for:** Sequential logic, branching (if/else), OS tasks.
* **Cores:** Few (4-64 powerful cores).

### The GPU (Graphics Processing Unit)
* **Analogy:** A Bus (or a fleet of buses).
* **Strength:** Slow at starting, but carries massive amounts of data at once (High Throughput).
* **Best for:** Parallel tasks (matrix math, image processing).
* **Cores:** Many (Thousands of smaller, weaker cores).

### Key Takeaway
**GPUs are not "faster" CPUs.** They are throughput engines. You should only use them when you can parallelize a task across thousands of data points.

# Block 2: CuPy - NumPy on Steroids (40 Mins)

**Objective:** Learn to use CuPy as a drop-in replacement for NumPy to accelerate matrix operations.

### What is CuPy?
CuPy is a library that implements the NumPy array interface but runs on NVIDIA GPUs. 

**The Golden Rule:** 
> `import numpy as np`  -->  `import cupy as cp`

In [ ]:
import numpy as np
import cupy as cp
import time

# 1. Array Creation
# Create a standard NumPy array (Exists in System RAM)
x_cpu = np.array([1, 2, 3])

# Create a CuPy array (Exists in GPU VRAM)
x_gpu = cp.array([1, 2, 3])

print("CPU Array Device:", x_cpu.dtype)
print("GPU Array Device:", x_gpu.device)

### ⚡ Live Benchmark: Matrix Multiplication
Let's multiply two large matrices (5000x5000) and compare speeds.

In [ ]:
# Configuration
SIZE = 5000

# --- CPU Benchmark ---
print(f"Initializing CPU arrays ({SIZE}x{SIZE})...")
a_cpu = np.random.rand(SIZE, SIZE)
b_cpu = np.random.rand(SIZE, SIZE)

print("Starting CPU Matrix Multiplication...")
start = time.time()
c_cpu = np.dot(a_cpu, b_cpu)
end = time.time()
cpu_time = end - start
print(f"CPU Time: {cpu_time:.4f} seconds")

# --- GPU Benchmark ---
print(f"\nInitializing GPU arrays ({SIZE}x{SIZE})...")
a_gpu = cp.random.rand(SIZE, SIZE)
b_gpu = cp.random.rand(SIZE, SIZE)

# Warmup pass (GPUs have initialization overhead)
cp.dot(a_gpu, b_gpu)
cp.cuda.Stream.null.synchronize() # Wait for GPU to finish

print("Starting GPU Matrix Multiplication...")
start = time.time()
c_gpu = cp.dot(a_gpu, b_gpu)
cp.cuda.Stream.null.synchronize() # CRITICAL: Wait for GPU
end = time.time()
gpu_time = end - start
print(f"GPU Time: {gpu_time:.4f} seconds")

# Speedup Calculation
print(f"\nSpeedup: {cpu_time / gpu_time:.1f}x faster")

### 📝 Guided Exercise 1
**Task:** Refactor the following CPU code to run on the GPU using CuPy.

```python
import numpy as np
def calculate_hypotenuse(a, b):
    return np.sqrt(a**2 + b**2)
```

1. Create two arrays of size 10,000,000 on the GPU.
2. Calculate the hypotenuse.
3. Measure the time.

In [ ]:
# TODO: Student Implementation Here

# Solution:
# a_gpu = cp.random.random(10000000)
# b_gpu = cp.random.random(10000000)
# start = time.time()
# res = cp.sqrt(a_gpu**2 + b_gpu**2)
# cp.cuda.Stream.null.synchronize()
# print(f"Done in {time.time()-start}s")

# Block 3: The Bottleneck - Data Transfer (15 Mins)

**Objective:** Understand that moving data between CPU (RAM) and GPU (VRAM) is expensive.

### The PCIe Bus
Think of the CPU and GPU as two islands connected by a bridge (PCIe bus). Computing on the island is fast, but crossing the bridge is slow.

**Common Pitfall:** Moving data back and forth inside a loop.

In [ ]:
# Demonstrating the Overhead
size_small = 100

# CPU
a_cpu = np.random.rand(size_small, size_small)

start = time.time()
np.dot(a_cpu, a_cpu)
print(f"CPU (Small Data): {time.time() - start:.6f}s")

# GPU (Includes Transfer Time)
start = time.time()
# 1. Transfer to GPU
a_gpu = cp.asarray(a_cpu) 
# 2. Compute
res_gpu = cp.dot(a_gpu, a_gpu)
# 3. Transfer back to CPU
res_cpu = cp.asnumpy(res_gpu)
print(f"GPU (Small Data + Transfer): {time.time() - start:.6f}s")

# Insight: For small data, the overhead > computation time.

--- 
## ☕ 15 Minute Break
---

# Block 5: Numba - Custom GPU Kernels (45 Mins)

**Objective:** Use Numba's JIT (Just-In-Time) compiler to write Python functions that compile to CUDA code.

### Why Numba?
CuPy is great for matrices, but what if you have complex logic (custom formulas, loops) that isn't just a matrix multiplication?

We will use the `@vectorize` decorator to create "Universal Functions" (ufuncs) that run on the GPU.

In [ ]:
from numba import vectorize, cuda
import math

# Define a function to compile for the GPU
# The signature 'float32(float32, float32)' means: Return float32, take two float32 inputs

@vectorize(['float32(float32, float32)'], target='cuda')
def gpu_function(x, y):
    # This looks like Python, but it compiles to a CUDA Kernel
    return math.sin(x) * math.cos(y) + math.exp(x / 100.0)

# Create data
n = 10000000
x = np.ones(n, dtype=np.float32)
y = np.ones(n, dtype=np.float32)

print("Running Numba CUDA Kernel...")
start = time.time()
# We can pass NumPy arrays directly; Numba handles the transfer (though manually managing is faster)
result = gpu_function(x, y)
print(f"Done in {time.time() - start:.4f}s")

### ⚠️ Important Concepts for Numba
1.  **Scalar Operations Only:** Inside the `@vectorize` function, you write logic for *one single element*. The GPU applies this to millions of elements at once.
2.  **No Python Objects:** You cannot use lists, dictionaries, or strings inside these functions. Only numbers.

# Block 6: Capstone Exercise - Monte Carlo Pi (30 Mins)

**Concept:** We can estimate the value of Pi by throwing random darts at a square board and counting how many land inside the inscribed circle.

**Task:**
1. Generate random `x` and `y` coordinates.
2. Calculate distance from center: $d = \sqrt{x^2 + y^2}$
3. Check if $d <= 1.0$.
4. Ratio of (inside / total) * 4 $\approx \pi$.

**Instructions:**
Write a solution using **CuPy** to perform this simulation with 100,000,000 points. Compare it to a pure Python implementation.

In [ ]:
# --- Student Workspace ---
# Hint: Use cp.random.rand()





In [ ]:
# --- Solution ---

def estimate_pi_gpu(num_points):
    print(f"Simulating {num_points} points on GPU...")
    start = time.time()
    
    # 1. Generate random numbers directly on GPU
    x = cp.random.rand(num_points, dtype=cp.float32)
    y = cp.random.rand(num_points, dtype=cp.float32)
    
    # 2. Compute distance (vectorized)
    dist = x**2 + y**2
    
    # 3. Count points inside circle
    inside_circle = (dist <= 1.0).sum()
    
    pi_est = 4.0 * inside_circle / num_points
    cp.cuda.Stream.null.synchronize()
    
    end = time.time()
    print(f"Estimated Pi: {pi_est}")
    print(f"Time taken: {end - start:.4f}s")
    return pi_est

estimate_pi_gpu(100000000)

# Block 7: Wrap-up & Best Practices (15 Mins)

## 🎓 Summary

1.  **Use CuPy** as your first line of defense. It's the easiest way to speed up NumPy code.
2.  **Use Numba** when you need custom element-wise logic that CuPy doesn't support.
3.  **Mind the Transfer:** Avoid `cp.asnumpy()` inside loops. Keep data on the GPU as long as possible.
4.  **Profile First:** Don't optimize code that isn't slow. Use `%time` or `time.time()`.

## 📚 Further Reading
* [CuPy Documentation](https://docs.cupy.dev/en/stable/)
* [Numba for CUDA GPUs](https://numba.readthedocs.io/en/stable/cuda/index.html)

**End of Session**